In [0]:
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("table_name","")
dbutils.widgets.text("merge_keys","")
dbutils.widgets.text("load_type","")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

print("=" * 80)
print("STEP 1: Reading input parameters")
print("=" * 80)
print(f"  landing_timestamp : '{landing_timestamp}'")
print(f"  table_name        : '{table_name}'")
print(f"  merge_keys        : '{merge_keys}'")
print(f"  load_type         : '{load_type}'")

In [0]:
source_table = f"adb_medlake.bronze.{table_name}"
filter_condition = f"landing_timestamp = '{landing_timestamp}'"

print("=" * 80)
print("STEP 2: Reading data from bronze layer")
print("=" * 80)
print(f"  Source table     : {source_table}")
print(f"  Filter condition : {filter_condition}")
print(f"  Equivalent query : SELECT * FROM {source_table} WHERE {filter_condition}")

bronze_df=spark.read.table(source_table).filter(filter_condition)

print(f"  Columns ({len(bronze_df.columns)}) : {bronze_df.columns}")
print(f"  Rows read from bronze : {bronze_df.count()}")

In [0]:
from pyspark.sql import functions as F
target_table = f"adb_medlake.silver.{table_name}"
normalized_load_type = (load_type or "FULL").strip().upper()
merge_key_list = [key.strip() for key in merge_keys.split(",") if key.strip()]

print("=" * 80)
print("STEP 3: Validating parameters and preparing data")
print("=" * 80)
print(f"  Target table         : {target_table}")
print(f"  Normalized load type : {normalized_load_type}")
print(f"  Parsed merge keys    : {merge_key_list}")

if normalized_load_type == "MERGE":
    if not merge_key_list:
        raise ValueError("merge_keys must be provided when load_type is 'MERGE'.")

    missing_merge_keys = [key for key in merge_key_list if key not in bronze_df.columns]
    if missing_merge_keys:
        raise ValueError(f"merge_keys not found in source data: {missing_merge_keys}")
    print("  Merge key validation passed - all keys exist in source data")

updated_df = (
    bronze_df.withColumn("insert_timestamp", F.current_timestamp())
    .withColumn("update_timestamp", F.current_timestamp())
)
print("  Added audit columns: insert_timestamp, update_timestamp")
print(f"  Final columns ({len(updated_df.columns)}) : {updated_df.columns}")

In [0]:
print("=" * 80)
print(f"STEP 4: Writing data to silver layer ({normalized_load_type})")
print("=" * 80)

if normalized_load_type == "FULL":
    print(f"  FULL load: overwriting table {target_table}")
    print(f"  Equivalent query : CREATE OR REPLACE TABLE {target_table} AS SELECT * FROM <updated_df>")
    updated_df.write.mode("overwrite").saveAsTable(target_table)
    print("  FULL load completed")

elif normalized_load_type == "APPEND":
    print(f"  APPEND load: appending rows to table {target_table}")
    print(f"  Equivalent query : INSERT INTO {target_table} SELECT * FROM <updated_df>")
    updated_df.write.mode("append").saveAsTable(target_table)
    print("  APPEND load completed")

elif normalized_load_type == "MERGE":
    if not spark.catalog.tableExists(target_table):
        print(f"  Target table {target_table} does not exist - creating it with an initial full load")
        updated_df.write.mode("overwrite").saveAsTable(target_table)
        print("  Initial load completed")
    else:
        print(f"  Target table {target_table} exists - performing MERGE")
        source_view = "silver_merge_source"
        updated_df.createOrReplaceTempView(source_view)
        print(f"  Created temp view '{source_view}' from source data")

        merge_condition = " AND ".join(
            [f"target.`{key}` = source.`{key}`" for key in merge_key_list]
        )

        update_assignments = ", ".join(
            [
                f"target.`{column}` = source.`{column}`"
                for column in updated_df.columns
                if column != "insert_timestamp"
            ]
        )

        insert_columns = ", ".join([f"`{column}`" for column in updated_df.columns])
        insert_values = ", ".join([f"source.`{column}`" for column in updated_df.columns])

        print(f"  Merge condition    : {merge_condition}")
        print(f"  Update assignments : {update_assignments}")
        print(f"  Insert columns     : {insert_columns}")
        print(f"  Insert values      : {insert_values}")

        merge_query = f"""
            MERGE INTO {target_table} AS target
            USING {source_view} AS source
            ON {merge_condition}
            WHEN MATCHED THEN UPDATE SET {update_assignments}
            WHEN NOT MATCHED THEN INSERT ({insert_columns}) VALUES ({insert_values})
        """
        print("  Prepared MERGE query:")
        print("-" * 80)
        print(merge_query)
        print("-" * 80)

        merge_result = spark.sql(merge_query)
        print("  MERGE completed. Result metrics:")
        for row in merge_result.collect():
            print(f"    {row.asDict()}")

else:
    raise ValueError("load_type must be one of: FULL, APPEND, MERGE")

In [0]:
record_count = updated_df.count()
print("=" * 80)
print("STEP 5: Finished")
print("=" * 80)
print(f"  Records processed from {source_table} to {target_table} : {record_count}")
dbutils.notebook.exit(str(record_count))